In [1]:
import os, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from rapidfuzz import fuzz

OUTPUT_DIR = '/Users/tm/Documents/GitHub/Project-for-Work/data/processed'
RANDOM_SEED = 42
THRESHOLD_HIGH = 0.90
THRESHOLD_MID  = 0.70
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- load artifacts ---
with open(f'{OUTPUT_DIR}/feature_cols.pkl','rb') as f: feature_cols = pickle.load(f)
with open(f'{OUTPUT_DIR}/scaler.pkl','rb') as f: scaler = pickle.load(f)
with open(f'{OUTPUT_DIR}/calibrator.pkl','rb') as f: calibrator = pickle.load(f)

class IdentityMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64,1)
        )
    def forward(self, x): return self.network(x).squeeze(-1)

model = IdentityMLP(len(feature_cols)).to(device)
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/model.pt', map_location=device, weights_only=True))
model.eval()

# --- load data ---
candidate_pairs = pd.read_csv(f'{OUTPUT_DIR}/candidate_pairs.csv')
df_clean = pd.read_csv(f'{OUTPUT_DIR}/all_profiles_cleaned.csv')

# profile_lookup
profile_lookup = df_clean.set_index('profile_id')
for col in ['userName_clean','fullName_clean','bio_clean','externalUrl_clean','url_domain','platform','location_clean']:
    if col not in profile_lookup.columns:
        profile_lookup[col] = ''

# TF-IDF for bio
bios = df_clean['bio_clean'].fillna('').astype(str).tolist()
tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
tfidf_matrix = tfidf_vec.fit_transform(bios)
key_to_idx = {pid: i for i, pid in enumerate(df_clean['profile_id'].tolist())}

def string_sim(a, b, method='jaro'):
    a, b = str(a) if a else '', str(b) if b else ''
    if method == 'jaro':        return fuzz.ratio(a, b) / 100
    if method == 'token_sort':  return fuzz.token_sort_ratio(a, b) / 100
    if method == 'levenshtein': return fuzz.ratio(a, b) / 100
    return 0.0

def get_decision(prob):
    if prob >= THRESHOLD_HIGH: return 'MATCH'
    elif prob >= THRESHOLD_MID: return 'POSSIBLE_MATCH'
    return 'NO_MATCH'

print(f'Artifacts loaded | candidate_pairs: {len(candidate_pairs):,} | profiles: {len(df_clean):,}')


Artifacts loaded | candidate_pairs: 15,467 | profiles: 24,729


---
## Stage 14: Identity Resolution Inference & Merge Decision

**วัตถุประสงค์:** รัน inference บน candidate pairs → ตัดสินใจ merge/review/reject

**Input:** `candidate_pairs`, `model.pt`, `scaler.pkl`, `calibrator.pkl`  
**Output:** `predictions.parquet` พร้อม 3-level decisions

| Sub-step | หน้าที่ |
|----------|--------|
| 14.1 | Load Artifacts |
| 14.2 | Compute Features + Predict |
| 14.3 | Apply 3-Level Threshold |
| 14.4 | Save Predictions |

### Step 14.1: Load Artifacts

### Step 14.2: Compute Features for Candidates + Predict

### Step 14.3-14.4: Apply 3-Level Threshold & Save

In [2]:

print("📊 Step 14.1: Load Artifacts")
print("=" * 60)
print(f"  Model       : IdentityMLP ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Features    : {len(feature_cols)} columns")
print(f"  Candidate pairs: {len(candidate_pairs):,}")
print(f"\n✅ Step 14.1 เสร็จ")


📊 Step 14.1: Load Artifacts
  Model       : IdentityMLP (46,721 params)
  Features    : 17 columns
  Candidate pairs: 15,467

✅ Step 14.1 เสร็จ


In [3]:

print("📊 Step 14.2: Inference on Candidate Pairs")
print("=" * 60)

infer_rows = []
for _, pair in candidate_pairs.iterrows():
    row = {}
    id_a, id_b = pair['profile_id_a'], pair['profile_id_b']
    if id_a in profile_lookup.index and id_b in profile_lookup.index:
        r_a = profile_lookup.loc[id_a]
        r_b = profile_lookup.loc[id_b]
        if isinstance(r_a, pd.DataFrame): r_a = r_a.iloc[0]
        if isinstance(r_b, pd.DataFrame): r_b = r_b.iloc[0]
        for col, prefix in [('userName_clean','username'),('fullName_clean','fullname'),('bio_clean','bio')]:
            va, vb = str(r_a.get(col,'') or ''), str(r_b.get(col,'') or '')
            for method in ['jaro','token_sort','levenshtein']:
                row[f'{prefix}_{method}'] = string_sim(va, vb, method)
            row[f'{prefix}_both_empty'] = 1.0 if (len(va)==0 and len(vb)==0) else 0.0
        idx_a = key_to_idx.get(id_a); idx_b = key_to_idx.get(id_b)
        if idx_a is not None and idx_b is not None:
            row['bio_tfidf_cosine'] = float(sk_cosine(tfidf_matrix[idx_a:idx_a+1], tfidf_matrix[idx_b:idx_b+1])[0][0])
        else:
            row['bio_tfidf_cosine'] = 0.0
        ua, ub = str(r_a.get('externalUrl_clean','') or ''), str(r_b.get('externalUrl_clean','') or '')
        da, db = str(r_a.get('url_domain','') or ''), str(r_b.get('url_domain','') or '')
        row['url_exact_match']  = 1.0 if (ua and ub and ua==ub) else 0.0
        row['url_domain_match'] = 1.0 if (da and db and da==db) else 0.0
        row['same_platform']    = 1.0 if r_a.get('platform','')==r_b.get('platform','') else 0.0
        la, lb = str(r_a.get('location_clean','') or ''), str(r_b.get('location_clean','') or '')
        row['location_sim'] = string_sim(la, lb, 'jaro') if (la and lb) else 0.0
    else:
        for col in feature_cols: row[col] = 0.0
    infer_rows.append(row)

infer_df = pd.DataFrame(infer_rows)
for col in feature_cols:
    if col not in infer_df.columns: infer_df[col] = 0.0

X_infer = scaler.transform(infer_df[feature_cols].values)
model.eval()
import torch
with torch.no_grad():
    raw_probs = torch.sigmoid(model(torch.FloatTensor(X_infer).to(device))).cpu().numpy()
cal_probs_infer = calibrator.predict(raw_probs)

print(f"  Predictions: {len(cal_probs_infer):,}")
print(f"  Prob mean={cal_probs_infer.mean():.4f}  std={cal_probs_infer.std():.4f}")
print(f"\n✅ Step 14.2 เสร็จ")


📊 Step 14.2: Inference on Candidate Pairs
  Predictions: 15,467
  Prob mean=0.7005  std=0.3983

✅ Step 14.2 เสร็จ


In [4]:

predictions_df = candidate_pairs.copy()
predictions_df['probability'] = cal_probs_infer
predictions_df['decision'] = predictions_df['probability'].apply(get_decision)

auto_merge   = predictions_df[predictions_df['decision']=='MATCH']
review_queue = predictions_df[predictions_df['decision']=='POSSIBLE_MATCH']
no_match     = predictions_df[predictions_df['decision']=='NO_MATCH']

print("=" * 60)
print("📊 STAGE 14 SUMMARY — Inference & Merge Decision")
print("=" * 60)
print(f"  Total      : {len(predictions_df):,}")
print(f"  MATCH      : {len(auto_merge):,} → auto-merge")
print(f"  POSSIBLE   : {len(review_queue):,} → human review")
print(f"  NO_MATCH   : {len(no_match):,} → keep separate")
if len(auto_merge) > 0:
    print(f"\n  Top 5 MATCH pairs:")
    for _, r in auto_merge.nlargest(5,'probability').iterrows():
        print(f"    prob={r['probability']:.3f} | {r['profile_id_a'][:35]} ↔ {r['profile_id_b'][:35]}")

predictions_df.to_csv('/Users/tm/Documents/GitHub/Project-for-Work/data/processed/predictions.csv', index=False)
print(f"\n  💾 Saved: predictions.csv")
print(f"\n{'='*60}")
print("✅ Stage 14 COMPLETE")
print(f"{'='*60}")


📊 STAGE 14 SUMMARY — Inference & Merge Decision
  Total      : 15,467
  MATCH      : 9,437 → auto-merge
  POSSIBLE   : 449 → human review
  NO_MATCH   : 5,581 → keep separate

  Top 5 MATCH pairs:
    prob=1.000 | zivgillat ↔ zivgillat
    prob=1.000 | eventosfera ↔ eventosfera
    prob=1.000 | theoryoferin ↔ theoryoferin
    prob=1.000 | theomeder ↔ theomeder
    prob=1.000 | eugenekaspersky ↔ eugenekaspersky

  💾 Saved: predictions.csv

✅ Stage 14 COMPLETE
